# 🍽️ Weekly Themed Meal Planner MVP

A meal planning assistant that generates 7 themed dinners with consolidated shopping lists.

## Features
- 7 unique themed dinners (Monday-Sunday)
- 5 ingredients per recipe (excluding basics)
- Dietary preference support (vegan, vegetarian, keto, omnivore)
- Consolidated shopping list
- Email delivery

## 1. Installation & Setup

In [1]:
%pip install -q langchain langchain-openai langchain-anthropic pydantic gradio python-dotenv
%pip install pyautogui pillow

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## 2. Configuration

Create a file called `.env` in the same folder as this notebook with this line:

```
OPENAI_API_KEY=sk-...
```

**Edit the values below before running the notebook.**

In [2]:
# ============================================================
# CONFIGURATION - Edit these values
# ============================================================
import os
from dotenv import load_dotenv

load_dotenv(".env")

# API Keys
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Model settings
OPENAI_MODEL = "gpt-4o-mini"
TEMPERATURE = 0.7

# Email settings (Gmail with App Password)
EMAIL_SENDER = "provide your email"
EMAIL_PASSWORD = "password"

# Meal generation settings
INGREDIENTS_PER_RECIPE = 5
EXCLUDED_BASICS = ["salt", "pepper", "oil", "garlic", "water"]
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]



# The "CONFIGURATION" cell loads API Key through .env, but it has no gitignore. When pushed to GitHub the API key will be visible.


In [3]:
# Set environment variables
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

## 3. Pydantic Models for Validation

In [4]:
from pydantic import BaseModel, Field, field_validator
from typing import List, Optional
from enum import Enum


class DietaryPreference(str, Enum):
    OMNIVORE = "omnivore"
    VEGETARIAN = "vegetarian"
    VEGAN = "vegan"
    KETO = "keto"


class Ingredient(BaseModel):
    """Single ingredient with quantity."""
    name: str = Field(..., description="Name of the ingredient")
    quantity: str = Field(..., description="Quantity with unit (e.g., '2 cups', '500g')")


class Recipe(BaseModel):
    """A single themed dinner recipe."""
    day: str = Field(..., description="Day of the week")
    theme: str = Field(..., description="Cuisine theme (e.g., 'Italian', 'Mexican')")
    dish_name: str = Field(..., description="Name of the dish")
    ingredients: List[Ingredient] = Field(..., description="List of 5 ingredients")
    instructions: str = Field(..., description="Brief cooking instructions")

    @field_validator('ingredients')
    @classmethod
    def validate_ingredient_count(cls, v):
        if len(v) != INGREDIENTS_PER_RECIPE:
            raise ValueError(f"Recipe must have exactly {INGREDIENTS_PER_RECIPE} ingredients, got {len(v)}")
        return v


class ShoppingItem(BaseModel):
    """Consolidated shopping list item."""
    name: str = Field(..., description="Ingredient name")
    total_quantity: str = Field(..., description="Total quantity needed")
    used_in: List[str] = Field(..., description="List of dishes using this ingredient")


class MealPlan(BaseModel):
    """Complete weekly meal plan."""
    dietary_preference: str = Field(..., description="Selected dietary preference")
    recipes: List[Recipe] = Field(..., description="List of 7 recipes for the week")
    shopping_list: List[ShoppingItem] = Field(..., description="Consolidated shopping list")

    @field_validator('recipes')
    @classmethod
    def validate_recipe_count(cls, v):
        if len(v) != 7:
            raise ValueError(f"Meal plan must have exactly 7 recipes, got {len(v)}")
        return v

    @field_validator('recipes')
    @classmethod
    def validate_unique_themes(cls, v):
        themes = [r.theme.lower() for r in v]
        if len(themes) != len(set(themes)):
            raise ValueError("All 7 themes must be unique")
        return v

    

## 4. LangChain Setup & Prompts

In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser


def get_llm():
    """Get the configured LLM."""
    return ChatOpenAI(model=OPENAI_MODEL, temperature=TEMPERATURE)


# Main meal plan generation prompt
MEAL_PLAN_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a professional meal planner. Generate a weekly meal plan with EXACTLY 7 themed dinners.

STRICT RULES - FOLLOW EXACTLY:
1. Each recipe MUST have EXACTLY 5 main ingredients (do NOT count salt, pepper, oil, garlic, or water)
2. ALL 7 THEMES MUST BE COMPLETELY DIFFERENT - no duplicates allowed! Use: Italian, Mexican, Japanese, Indian, Thai, Greek, American, Chinese, French, Korean, Mediterranean, Middle Eastern, etc.
3. All ingredients MUST be real, commonly available items
4. Follow the dietary preference strictly
5. Provide realistic quantities for 2 servings

Dietary Guidelines:
- Omnivore: Any ingredients allowed
- Vegetarian: No meat or fish, eggs and dairy OK
- Vegan: No animal products at all
- Keto: Low carb, high fat, no grains/sugar/starchy vegetables

Output valid JSON matching this exact structure:
{format_instructions}"""),
    ("human", """Create a weekly meal plan:
- Diet: {dietary_preference}
- Theme requests: {theme_requests}
- Preferences: {additional_preferences}

IMPORTANT: Each day MUST have a DIFFERENT cuisine theme. Generate 7 unique themed dinners for Monday-Sunday.""")
])


# Shopping list consolidation prompt
SHOPPING_LIST_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a shopping list organizer. Consolidate ingredients from multiple recipes into a single shopping list.

RULES:
1. Combine same ingredients and sum their quantities
2. List which dishes use each ingredient
3. Use standard units (cups, tablespoons, grams, pieces)
4. Round up quantities for convenience

Output valid JSON matching this structure:
{format_instructions}"""),
    ("human", """Consolidate the shopping list from these recipes:
{recipes_json}""")
])

## 5. Meal Generation Logic

In [ ]:
import json
import time


class MealPlanGenerator:
    """Generates weekly themed meal plans."""

    def __init__(self):
        self.llm = get_llm()
        self.recipe_parser = JsonOutputParser(pydantic_object=MealPlan)
        self.shopping_parser = JsonOutputParser(pydantic_object=ShoppingItem)

    def generate(
        self,
        dietary_preference: str,
        theme_requests: str = "random",
        additional_preferences: str = "none"
    ) -> dict:
        """Generate a complete meal plan with shopping list."""
        start_time = time.time()

        recipe_chain = MEAL_PLAN_PROMPT | self.llm

        result = recipe_chain.invoke({
            "dietary_preference": dietary_preference,
            "theme_requests": theme_requests if theme_requests else "random diverse cuisines",
            "additional_preferences": additional_preferences if additional_preferences else "none",
            "format_instructions": self.recipe_parser.get_format_instructions()
        })

        try:
            content = result.content
            if "```json" in content:
                content = content.split("```json")[1].split("```")[0]
            elif "```" in content:
                content = content.split("```")[1].split("```")[0]

            meal_plan_data = json.loads(content)
            meal_plan = MealPlan(**meal_plan_data)
            generation_time = time.time() - start_time

            return {
                "success": True,
                "meal_plan": meal_plan.model_dump(),
                "generation_time": round(generation_time, 2),
                "error": None
            }

        except json.JSONDecodeError as e:
            return {"success": False, "meal_plan": None, "error": f"JSON parsing error: {str(e)}"}
        except Exception as e:
            return {"success": False, "meal_plan": None, "error": str(e)}




# 1
# The Meal Generator does 9 seperate jobs:

# Starts a timer  
# Builds the LLM chain  
# Applies default input values  
# Calls the LLM  
# Removes Markdown code fences  
# Parses the JSON  
# Validates it with MealPlan  
# Builds the success response  
# Handles errors and builds failure responses

# That makes it monolithic, because one "generate()" handles many unrelated responsibilities.
# Refactored version would spit into smaller functions (comms, input prep, validation, response formatting)

In [ ]:
# Improved generate()

class MealPlanGenerator:
    def generate(self, dietary_preference, theme_requests="random",
                 additional_preferences="none"):
        start_time = time.time()

        try:
            response = self._call_llm(
                dietary_preference,
                theme_requests,
                additional_preferences
            )
            meal_plan = self._parse_meal_plan(response)

            return {
                "success": True,
                "meal_plan": meal_plan.model_dump(),
                "generation_time": round(time.time() - start_time, 2),
                "error": None,
            }

        except Exception as exc:
            return {
                "success": False,
                "meal_plan": None,
                "generation_time": round(time.time() - start_time, 2),
                "error": str(exc),
            }

    def _call_llm(self, dietary_preference, theme_requests,
                  additional_preferences):
        chain = MEAL_PLAN_PROMPT | self.llm
        return chain.invoke({
            "dietary_preference": dietary_preference,
            "theme_requests": theme_requests or "random diverse cuisines",
            "additional_preferences": additional_preferences or "none",
            "format_instructions": self.recipe_parser.get_format_instructions(),
        })

    def _parse_meal_plan(self, response):
        content = response.content
        content = self._remove_code_fences(content)
        return MealPlan(**json.loads(content))

    def _remove_code_fences(self, content):
        if "```json" in content:
            return content.split("```json", 1)[1].split("```", 1)[0]
        if "```" in content:
            return content.split("```", 1)[1].split("```", 1)[0]
        return content

# 2
# recipe_chain.invoke() 
# recipe_chain.invoke() is outzide the "try" block
# With wrong API key it gives error before execution reaches "try".
# Will not be caught by "except"
# "generate()" stops
# Does not give 'succes' 'false'
# API Call should be moved inside the "try" block, or handled with dedicated helper with error reporting.

In [ ]:
# Improved recipe_chain.invoke, 
# API call moved inside "try" block

def generate(
    self,
    dietary_preference: str,
    theme_requests: str = "random",
    additional_preferences: str = "none"
) -> dict:
    """Generate a complete meal plan with shopping list."""
    start_time = time.time()

    try:
        recipe_chain = MEAL_PLAN_PROMPT | self.llm

        result = recipe_chain.invoke({
            "dietary_preference": dietary_preference,
            "theme_requests": (
                theme_requests
                if theme_requests
                else "random diverse cuisines"
            ),
            "additional_preferences": (
                additional_preferences
                if additional_preferences
                else "none"
            ),
            "format_instructions": (
                self.recipe_parser.get_format_instructions()
            )
        })

        content = result.content

        if "```json" in content:
            content = content.split("```json")[1].split("```")[0]
        elif "```" in content:
            content = content.split("```")[1].split("```")[0]

        meal_plan_data = json.loads(content)
        meal_plan = MealPlan(**meal_plan_data)
        generation_time = time.time() - start_time

        return {
            "success": True,
            "meal_plan": meal_plan.model_dump(),
            "generation_time": round(generation_time, 2),
            "error": None
        }

    except json.JSONDecodeError as e:
        return {
            "success": False,
            "meal_plan": None,
            "generation_time": round(time.time() - start_time, 2),
            "error": f"JSON parsing error: {str(e)}"
        }

    except Exception as e:
        return {
            "success": False,
            "meal_plan": None,
            "generation_time": round(time.time() - start_time, 2),
            "error": str(e)
        }

# 4

# "except" blocks
# Error message explains 'what' happende, but not 'where'.
# The exceptions lack location/context information, making debugging harder.

In [ ]:
# Improved recipe_chain.invoke, 
# API call moved inside "try" block
# Improved error handling for JSON parsing and API call failures

def generate(
    self,
    dietary_preference: str,
    theme_requests: str = "random",
    additional_preferences: str = "none"
) -> dict:
    """Generate a complete meal plan with shopping list."""
    start_time = time.time()

    try:
        recipe_chain = MEAL_PLAN_PROMPT | self.llm

        result = recipe_chain.invoke({
            "dietary_preference": dietary_preference,
            "theme_requests": (
                theme_requests
                if theme_requests
                else "random diverse cuisines"
            ),
            "additional_preferences": (
                additional_preferences
                if additional_preferences
                else "none"
            ),
            "format_instructions": (
                self.recipe_parser.get_format_instructions()
            )
        })

        content = result.content

        if "```json" in content:
            content = content.split("```json")[1].split("```")[0]
        elif "```" in content:
            content = content.split("```")[1].split("```")[0]

        meal_plan_data = json.loads(content)
        meal_plan = MealPlan(**meal_plan_data)
        generation_time = time.time() - start_time

        return {
            "success": True,
            "meal_plan": meal_plan.model_dump(),
            "generation_time": round(generation_time, 2),
            "error": None
        }

    except json.JSONDecodeError as e:
    return {
        "success": False,
        "meal_plan": None,
        "error": f"Meal-plan response parsing failed: {e}"
    }

except Exception as e:
    return {
        "success": False,
        "meal_plan": None,
        "error": f"Meal-plan generation/API call failed: {e}"
    }

# 5
# Fragile logic
# Fence stripping here is fragile because of its assumed response format.
# If the model writes a sentence before unfenced JSON, json.loads(content) fails.
# If the closing ``` fence is missing, extra text may remain and JSON parsing can fail.
# If the model uses different Markdown formatting or adds text after the JSON, the .split() logic may extract invalid content.
# If the expected markers are malformed, it can also raise an IndexError.

# Code does not safely extract and validate the JSON, but assumes clean code fences.

In [ ]:
# #5 fix
# This cell handles explanatory text before the JSON, optional code fences, and missing or invalid JSON more clearly.

def extract_json(text: str) -> dict:
    """Extract the first JSON object from an LLM response."""
    text = text.strip()

    # Remove optional Markdown code fences
    text = text.replace("```json", "").replace("```", "").strip()

    # Ignore any explanatory text before the JSON object
    start = text.find("{")

    if start == -1:
        raise ValueError("No JSON object found in the model response.")

    try:
        meal_plan_data, _ = json.JSONDecoder().raw_decode(text[start:])
        return meal_plan_data
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid meal-plan JSON: {e}") from e


class MealPlanGenerator:
    """Generates weekly themed meal plans."""

    def __init__(self):
        self.llm = get_llm()
        self.recipe_parser = JsonOutputParser(
            pydantic_object=MealPlan
        )
        self.shopping_parser = JsonOutputParser(
            pydantic_object=ShoppingItem
        )

    def generate(
        self,
        dietary_preference: str,
        theme_requests: str = "random",
        additional_preferences: str = "none"
    ) -> dict:
        """Generate a complete meal plan with shopping list."""
        start_time = time.time()

        try:
            recipe_chain = MEAL_PLAN_PROMPT | self.llm

            result = recipe_chain.invoke({
                "dietary_preference": dietary_preference,
                "theme_requests": (
                    theme_requests
                    if theme_requests
                    else "random diverse cuisines"
                ),
                "additional_preferences": (
                    additional_preferences
                    if additional_preferences
                    else "none"
                ),
                "format_instructions": (
                    self.recipe_parser.get_format_instructions()
                )
            })

            meal_plan_data = extract_json(result.content)
            meal_plan = MealPlan(**meal_plan_data)

            return {
                "success": True,
                "meal_plan": meal_plan.model_dump(),
                "generation_time": round(
                    time.time() - start_time,
                    2
                ),
                "error": None
            }

        except json.JSONDecodeError as e:
            return {
                "success": False,
                "meal_plan": None,
                "generation_time": round(
                    time.time() - start_time,
                    2
                ),
                "error": f"JSON parsing failed: {e}"
            }

        except Exception as e:
            return {
                "success": False,
                "meal_plan": None,
                "generation_time": round(
                    time.time() - start_time,
                    2
                ),
                "error": f"Meal-plan generation failed: {e}"
            }


# Initialize generator
generator = MealPlanGenerator()

# 6 Inconsistent output

# Code has insconsistency between "success" dictionary and "failure" dictionary:
# "generation time" included in success, not in failure.

In [ ]:
import json
import time


def extract_json(text: str) -> dict:
    """Extract the first JSON object from an LLM response."""
    text = text.strip()

    # Remove optional Markdown code fences
    text = text.replace("```json", "").replace("```", "").strip()

    # Ignore explanatory text before the JSON object
    start = text.find("{")

    if start == -1:
        raise ValueError("No JSON object found in the model response.")

    try:
        meal_plan_data, _ = json.JSONDecoder().raw_decode(text[start:])
        return meal_plan_data
    except json.JSONDecodeError as e:
        raise ValueError(f"Invalid meal-plan JSON: {e}") from e


class MealPlanGenerator:
    """Generates weekly themed meal plans."""

    def __init__(self):
        self.llm = get_llm()
        self.recipe_parser = JsonOutputParser(
            pydantic_object=MealPlan
        )
        self.shopping_parser = JsonOutputParser(
            pydantic_object=ShoppingItem
        )

    def generate(
        self,
        dietary_preference: str,
        theme_requests: str = "random",
        additional_preferences: str = "none"
    ) -> dict:
        """Generate a complete meal plan with shopping list."""
        start_time = time.time()

        try:
            # Build the chain
            recipe_chain = MEAL_PLAN_PROMPT | self.llm

            # Call the LLM inside the try block
            result = recipe_chain.invoke({
                "dietary_preference": dietary_preference,
                "theme_requests": (
                    theme_requests
                    if theme_requests
                    else "random diverse cuisines"
                ),
                "additional_preferences": (
                    additional_preferences
                    if additional_preferences
                    else "none"
                ),
                "format_instructions": (
                    self.recipe_parser.get_format_instructions()
                )
            })

            # Extract and validate the response
            meal_plan_data = extract_json(result.content)
            meal_plan = MealPlan(**meal_plan_data)

            # Return a consistent success response
            return {
                "success": True,
                "meal_plan": meal_plan.model_dump(),
                "generation_time": round(
                    time.time() - start_time,
                    2
                ),
                "error": None
            }

        except ValueError as e:
            return {
                "success": False,
                "meal_plan": None,
                "generation_time": round(
                    time.time() - start_time,
                    2
                ),
                "error": f"Response parsing failed: {e}"
            }

        except Exception as e:
            return {
                "success": False,
                "meal_plan": None,
                "generation_time": round(
                    time.time() - start_time,
                    2
                ),
                "error": f"Meal-plan generation failed: {e}"
            }


# Initialize generator
generator = MealPlanGenerator()

# 7 Repeated code
# Which line turns ingredients into text in both "email" and "recipe"

# Both functions repeat line
# ingredients = ", ".join(
#     [f"{ing['quantity']} {ing['name']}" for ing in recipe['ingredients']]
# )
#
# Fix with helper function:

# def format_ingredients(ingredients: list) -> str:
#    return ", ".join(
#        f"{ingredient['quantity']} {ingredient['name']}"
#        for ingredient in ingredients
#    )



# 8 
# Repeated code + bug
# Q: What is copy-pasted between them? And which plan does the email button actually send?

# A: Both functions copy the same meal plan generation coed

# They also dupclicate the failure check
# gives bug in send_email_ui*

# Fix: Take out duplicates
# Or
# def generate_plan_streaming(
    dietary_pref,
    theme_requests,
    additional_prefs
):
    """Generate and display one meal plan."""

    yield "Generating your meal plan...", None

    result = generator.generate(
        dietary_preference=dietary_pref,
        theme_requests=(
            theme_requests.strip()
            if theme_requests.strip()
            else "random"
        ),
        additional_preferences=(
            additional_prefs.strip()
            if additional_prefs.strip()
            else "none"
        )
    )

    if not result["success"]:
        yield f"Error: {result['error']}", None
        return

    meal_plan = result["meal_plan"]

    output = (
        f"## Weekly "
        f"{meal_plan['dietary_preference'].title()} Meal Plan\n\n"
    )
    output += format_shopping_list(meal_plan)
    output += "\n\n---\n\n## Recipes\n\n"

    yield output, meal_plan

    for recipe in meal_plan["recipes"]:
        output += format_single_recipe(recipe) + "\n"
        yield output, meal_plan

    validation = run_basic_validation(result)
    status = (
        "All checks passed"
        if validation["passed"]
        else f"Issues: {', '.join(validation['errors'])}"
    )

    output += (
        f"---\n"
        f"*Generated in {result['generation_time']}s | {status}*"
    )

    yield output, meal_plan


def send_email_ui(email, meal_plan):
    """Email the currently displayed meal plan."""

    if not email or "@" not in email:
        return "Enter a valid email address."

    if not meal_plan:
        return "Generate a meal plan before emailing it."

    email_result = send_meal_plan_email(email, meal_plan)
    return email_result["message"]

# Then update the Gradio interface

with gr.Blocks(title="Weekly Meal Planner") as app:
    meal_plan_state = gr.State(value=None)

    # Existing components ...

    generate_btn.click(
        fn=generate_plan_streaming,
        inputs=[
            dietary_dropdown,
            theme_input,
            additional_input
        ],
        outputs=[
            output_display,
            meal_plan_state
        ]
    )

    send_btn.click(
        fn=send_email_ui,
        inputs=[
            email_input,
            meal_plan_state
        ],
        outputs=email_status
    )

# working email button



# 9 Dead Code

# Never used:

| Item | Used? | Explanation |
|---|---:|---|
| `MEAL_PLAN_PROMPT` | Yes | Used to build `recipe_chain` in `generate()`. |
| `SHOPPING_LIST_PROMPT` | No | Defined in the prompts cell but never invoked. |
| `recipe_parser` | Yes | Supplies format instructions for the meal-plan response. |
| `shopping_parser` | No | Created in `__init__`, but never used to parse anything. |
| `MealPlan` | Yes | Validates the generated meal plan. |
| `ShoppingItem` | Yes, indirectly | Used as the type for items inside `MealPlan.shopping_list`. |

# Dead code should either be removed or used in seperate shopping-list generation


# 10 Hard coded values

# The numbers 5 and 7 appear here. Is there already a constant for them?

# Answer is yes, run_basic_validation() hardcodes values that already exist elsewhere

# 5 should use INGREDIENTS_PER_RECIPE
# 7 should use len(days)

def run_basic_validation(meal_plan_result: dict) -> dict:
    """Run quick programmatic validation checks."""
    if not meal_plan_result.get("success"):
        return {
            "passed": False,
            "errors": [meal_plan_result.get("error")]
        }

    meal_plan = meal_plan_result["meal_plan"]
    errors = []

    expected_recipe_count = len(DAYS)

    # Check recipe count
    if len(meal_plan["recipes"]) != expected_recipe_count:
        errors.append(
            f"Expected {expected_recipe_count} recipes, "
            f"got {len(meal_plan['recipes'])}"
        )

    # Check unique themes
    themes = [
        recipe["theme"].lower()
        for recipe in meal_plan["recipes"]
    ]

    if len(themes) != len(set(themes)):
        duplicates = [
            theme for theme in themes
            if themes.count(theme) > 1
        ]
        errors.append(f"Duplicate themes: {set(duplicates)}")

    # Check ingredients per recipe
    for recipe in meal_plan["recipes"]:
        ingredient_count = len(recipe["ingredients"])

        if ingredient_count != INGREDIENTS_PER_RECIPE:
            errors.append(
                f"{recipe['day']}: {ingredient_count} ingredients "
                f"(expected {INGREDIENTS_PER_RECIPE})"
            )

    return {
        "passed": len(errors) == 0,
        "errors": errors
    }

# 11 Duplicated Logic

How many places check the same rules?

# Same 3 rules are checked in 2 places.
    Pydantic models:
        recipe.validate_ingredient_count()
        MealPlan.validate_recipe_count()
        MealPlan.validate_unique_themes()
    run_basic_validation*()
        Checks the number of ingredients
        checks number of recipes
        Checks for duplicate themes

# maintenenace risk, ipmlementation could become inconsisten

# Fix: Use the Pydantic models as the single source of truth and remove the duplicated manual checks from run_basic_validation*(


from pydantic import ValidationError


def run_basic_validation(meal_plan_result: dict) -> dict:
    """Validate a meal plan using the Pydantic models."""

    if not meal_plan_result.get("success"):
        return {
            "passed": False,
            "errors": [meal_plan_result.get("error")]
        }

    try:
        # Reuse the existing Pydantic validation rules
        MealPlan(**meal_plan_result["meal_plan"])

        return {
            "passed": True,
            "errors": []
        }

    except ValidationError as e:
        return {
            "passed": False,
            "errors": [
                error["msg"]
                for error in e.errors()
            ]
        }
)

# Also update the Pydantic recipe-count validator to use the existing "DAYS" constant:


@field_validator("recipes")
@classmethod
def validate_recipe_count(cls, recipes):
    expected_count = len(DAYS)

    if len(recipes) != expected_count:
        raise ValueError(
            f"Meal plan must have exactly "
            f"{expected_count} recipes, got {len(recipes)}"
        )

    return recipes

Now the validation rules live only in the Pydantic models, while run_basic_validation() simply reuses them.

# 12 Mixed concerns

# Q: Building a message and sending it over the network: one job or two?
# A: Two


# 13 Mixed concerns

# Q: Test data, the loop and the printing all live together. Could the test cases come from a file instead?

# A: Yes, mixed 3 responsibilities:
# Defining test data
# Running each test and making real API call
# Printing the results


In [ ]:
### FINAL FULL REFACTOR OF ORIGINAL SCRIPT

# Cell 1: Configuration and imports

import json
import os
import smtplib
import time

from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from typing import List

import gradio as gr
from dotenv import load_dotenv
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field, ValidationError, field_validator


load_dotenv(".env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
TEMPERATURE = 0.7

EMAIL_SENDER = os.getenv("EMAIL_SENDER")
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")
SMTP_HOST = "smtp.gmail.com"
SMTP_PORT = 587

DAYS = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday",
]

INGREDIENTS_PER_RECIPE = 5

In [ ]:
# CELL 2: DATA MODELS


class Ingredient(BaseModel):
    name: str
    quantity: str


class Recipe(BaseModel):
    day: str
    theme: str
    dish_name: str
    ingredients: List[Ingredient]
    instructions: str

    @field_validator("ingredients")
    @classmethod
    def validate_ingredient_count(cls, ingredients):
        if len(ingredients) != INGREDIENTS_PER_RECIPE:
            raise ValueError(
                f"Recipe must contain exactly "
                f"{INGREDIENTS_PER_RECIPE} ingredients."
            )
        return ingredients


class ShoppingItem(BaseModel):
    name: str
    total_quantity: str
    used_in: List[str]


class MealPlan(BaseModel):
    dietary_preference: str
    recipes: List[Recipe]
    shopping_list: List[ShoppingItem]

    @field_validator("recipes")
    @classmethod
    def validate_recipe_count(cls, recipes):
        expected_count = len(DAYS)

        if len(recipes) != expected_count:
            raise ValueError(
                f"Meal plan must contain exactly "
                f"{expected_count} recipes."
            )
        return recipes

    @field_validator("recipes")
    @classmethod
    def validate_unique_themes(cls, recipes):
        themes = [recipe.theme.lower() for recipe in recipes]

        if len(themes) != len(set(themes)):
            raise ValueError("All recipe themes must be unique.")

        return recipes

In [ ]:
# Cell 3: LLM setup and prompt

def get_llm():
    return ChatOpenAI(
        api_key=OPENAI_API_KEY,
        model=OPENAI_MODEL,
        temperature=TEMPERATURE,
    )


MEAL_PLAN_PROMPT = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a professional meal planner.

Generate a weekly meal plan with exactly 7 themed dinners.

Rules:
- Generate exactly 7 recipes.
- Use a different cuisine theme for every recipe.
- Each recipe must contain exactly 5 main ingredients.
- Follow the dietary preference strictly.
- Provide realistic quantities for 2 servings.
- Return valid JSON only.

The JSON must match this structure:
{format_instructions}
"""
    ),
    (
        "human",
        """Create a weekly meal plan.

Dietary preference: {dietary_preference}
Theme requests: {theme_requests}
Additional preferences: {additional_preferences}
"""
    ),
])

In [ ]:
# Cell 4: Response parsing and meal generation

def extract_json(text: str) -> dict:
    """Extract the first JSON object from an LLM response."""

    cleaned_text = (
        text
        .replace("```json", "")
        .replace("```", "")
        .strip()
    )

    start_index = cleaned_text.find("{")

    if start_index == -1:
        raise ValueError(
            "No JSON object was found in the LLM response."
        )

    try:
        data, _ = json.JSONDecoder().raw_decode(
            cleaned_text[start_index:]
        )
        return data

    except json.JSONDecodeError as error:
        raise ValueError(
            f"Invalid JSON in LLM response: {error}"
        ) from error


class MealPlanGenerator:
    """Generates and validates weekly meal plans."""

    def __init__(self):
        self.llm = get_llm()
        self.parser = JsonOutputParser(
            pydantic_object=MealPlan
        )

    def _build_request(
        self,
        dietary_preference: str,
        theme_requests: str,
        additional_preferences: str,
    ) -> dict:
        return {
            "dietary_preference": dietary_preference,
            "theme_requests": (
                theme_requests.strip()
                if theme_requests.strip()
                else "random diverse cuisines"
            ),
            "additional_preferences": (
                additional_preferences.strip()
                if additional_preferences.strip()
                else "none"
            ),
            "format_instructions": (
                self.parser.get_format_instructions()
            ),
        }

    def _call_llm(self, request: dict):
        chain = MEAL_PLAN_PROMPT | self.llm
        return chain.invoke(request)

    def _parse_response(self, response) -> MealPlan:
        data = extract_json(response.content)
        return MealPlan(**data)

    def generate(
        self,
        dietary_preference: str,
        theme_requests: str = "random",
        additional_preferences: str = "none",
    ) -> dict:
        """Generate one validated meal plan."""

        start_time = time.time()

        try:
            request = self._build_request(
                dietary_preference,
                theme_requests,
                additional_preferences,
            )

            # API call is inside the try block.
            response = self._call_llm(request)
            meal_plan = self._parse_response(response)

            return {
                "success": True,
                "meal_plan": meal_plan.model_dump(),
                "generation_time": round(
                    time.time() - start_time,
                    2,
                ),
                "error": None,
            }

        except ValueError as error:
            return {
                "success": False,
                "meal_plan": None,
                "generation_time": round(
                    time.time() - start_time,
                    2,
                ),
                "error": f"Response parsing failed: {error}",
            }

        except Exception as error:
            return {
                "success": False,
                "meal_plan": None,
                "generation_time": round(
                    time.time() - start_time,
                    2,
                ),
                "error": f"Meal generation/API call failed: {error}",
            }


generator = MealPlanGenerator()

In [ ]:
# Cell 5: Validation without duplicated rules

def run_basic_validation(meal_plan_result: dict) -> dict:
    """Reuse Pydantic validation instead of duplicating rules."""

    if not meal_plan_result.get("success"):
        return {
            "passed": False,
            "errors": [meal_plan_result.get("error")],
        }

    try:
        MealPlan(**meal_plan_result["meal_plan"])

        return {
            "passed": True,
            "errors": [],
        }

    except ValidationError as error:
        return {
            "passed": False,
            "errors": [
                validation_error["msg"]
                for validation_error in error.errors()
            ],
        }

In [ ]:
# Cell 6: Formatting helpers

def format_ingredients(ingredients: list) -> str:
    """Convert ingredients into readable text."""

    return ", ".join(
        f"{ingredient['quantity']} {ingredient['name']}"
        for ingredient in ingredients
    )


def format_shopping_list(meal_plan: dict) -> str:
    lines = ["## Shopping List\n"]

    for item in meal_plan["shopping_list"]:
        lines.append(
            f"- {item['total_quantity']} {item['name']}"
        )

    return "\n".join(lines)


def format_single_recipe(recipe: dict) -> str:
    ingredients = format_ingredients(
        recipe["ingredients"]
    )

    return (
        f"**{recipe['day']}** | "
        f"{recipe['theme']} | "
        f"*{recipe['dish_name']}*\n"
        f"{ingredients}\n"
    )

In [ ]:
# Cell 7: Email formatting and sending separated

def format_meal_plan_email(meal_plan: dict) -> str:
    """Build the readable email body."""

    lines = [
        "=" * 40,
        f"WEEKLY MEAL PLAN "
        f"({meal_plan['dietary_preference'].upper()})",
        "=" * 40,
        "",
        "SHOPPING LIST",
        "-" * 20,
    ]

    for item in meal_plan["shopping_list"]:
        lines.append(
            f"- {item['total_quantity']} {item['name']}"
        )

    lines.extend([
        "",
        "=" * 40,
        "RECIPES",
        "=" * 40,
    ])

    for recipe in meal_plan["recipes"]:
        lines.extend([
            f"\n{recipe['day'].upper()} - {recipe['theme']}",
            recipe["dish_name"],
            f"Ingredients: "
            f"{format_ingredients(recipe['ingredients'])}",
            f"Instructions: {recipe['instructions']}",
        ])

    return "\n".join(lines)


def build_email_message(
    recipient_email: str,
    meal_plan: dict,
) -> MIMEMultipart:
    """Build an email message without sending it."""

    message = MIMEMultipart()
    message["From"] = EMAIL_SENDER
    message["To"] = recipient_email
    message["Subject"] = "Your Weekly Meal Plan"

    body = format_meal_plan_email(meal_plan)
    message.attach(MIMEText(body, "plain"))

    return message


def send_email_message(message: MIMEMultipart) -> None:
    """Send an already-built message over SMTP."""

    with smtplib.SMTP(SMTP_HOST, SMTP_PORT) as server:
        server.starttls()
        server.login(EMAIL_SENDER, EMAIL_PASSWORD)
        server.send_message(message)

In [ ]:
# Cell 8: UI functions

def generate_plan_streaming(
    dietary_preference,
    theme_requests,
    additional_preferences,
):
    """Generate one plan and store that same plan for emailing."""

    yield "Generating your meal plan...", None

    result = generator.generate(
        dietary_preference=dietary_preference,
        theme_requests=theme_requests,
        additional_preferences=additional_preferences,
    )

    if not result["success"]:
        yield f"Error: {result['error']}", None
        return

    meal_plan = result["meal_plan"]

    output = (
        f"## Weekly "
        f"{meal_plan['dietary_preference'].title()} Meal Plan\n\n"
    )
    output += format_shopping_list(meal_plan)
    output += "\n\n---\n\n## Recipes\n\n"

    yield output, meal_plan

    for recipe in meal_plan["recipes"]:
        output += format_single_recipe(recipe)
        output += "\n"
        yield output, meal_plan

    validation = run_basic_validation(result)

    status = (
        "All checks passed"
        if validation["passed"]
        else f"Issues: {', '.join(validation['errors'])}"
    )

    output += (
        f"---\n"
        f"*Generated in {result['generation_time']}s | "
        f"{status}*"
    )

    yield output, meal_plan


def send_email_ui(email: str, meal_plan: dict):
    """Email the currently displayed plan."""

    if not email or "@" not in email:
        return "Enter a valid email address."

    if not meal_plan:
        return "Generate a meal plan before emailing it."

    try:
        message = build_email_message(email, meal_plan)
        send_email_message(message)

        return f"Email sent to {email}"

    except Exception as error:
        return f"Email sending failed: {error}"

In [ ]:
# Cell 9: Gradio interface

with gr.Blocks(title="Weekly Meal Planner") as app:
    meal_plan_state = gr.State(value=None)

    gr.Markdown(
        "# Weekly Meal Planner\n"
        "Generate 7 themed dinners with a shopping list."
    )

    with gr.Row():
        with gr.Column(scale=1):
            dietary_dropdown = gr.Dropdown(
                choices=[
                    "omnivore",
                    "vegetarian",
                    "vegan",
                    "keto",
                ],
                value="omnivore",
                label="Diet",
            )

            theme_input = gr.Textbox(
                label="Themes",
                placeholder="Italian, Mexican...",
            )

            additional_input = gr.Textbox(
                label="Additional preferences",
                placeholder="No mushrooms, high protein...",
            )

            generate_button = gr.Button(
                "Generate",
                variant="primary",
            )

            email_input = gr.Textbox(
                label="Email",
                placeholder="you@example.com",
            )

            send_button = gr.Button("Email Plan")

            email_status = gr.Textbox(
                label="Status",
                interactive=False,
            )

        with gr.Column(scale=2):
            output_display = gr.Markdown(
                "Click **Generate** to start."
            )

    generate_button.click(
        fn=generate_plan_streaming,
        inputs=[
            dietary_dropdown,
            theme_input,
            additional_input,
        ],
        outputs=[
            output_display,
            meal_plan_state,
        ],
    )

    send_button.click(
        fn=send_email_ui,
        inputs=[
            email_input,
            meal_plan_state,
        ],
        outputs=email_status,
    )

In [ ]:
# Cell 10: External test data and test runner

def run_test_cases(test_cases: list) -> list:
    """Run test cases and return structured results."""

    results = []

    for case in test_cases:
        result = generator.generate(**case)
        validation = run_basic_validation(result)

        results.append({
            "dietary_preference": case["dietary_preference"],
            "passed": validation["passed"],
            "errors": validation["errors"],
        })

    return results


with open("test_cases.json", "r") as file:
    test_cases = json.load(file)

test_results = run_test_cases(test_cases)

for test in test_results:
    status = (
        "PASS"
        if test["passed"]
        else f"FAIL: {test['errors']}"
    )

    print(
        f"{test['dietary_preference']} -> {status}"
    )

In [ ]:
JSON
[
  {
    "dietary_preference": "omnivore",
    "theme_requests": "random",
    "additional_preferences": "none"
  },
  {
    "dietary_preference": "vegetarian",
    "theme_requests": "Italian, Asian, Mexican",
    "additional_preferences": "no mushrooms"
  },
  {
    "dietary_preference": "vegan",
    "theme_requests": "random",
    "additional_preferences": "high protein"
  },
  {
    "dietary_preference": "keto",
    "theme_requests": "Mediterranean, American",
    "additional_preferences": "none"
  }
]

## 6. Email Functionality

In [7]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart


def format_meal_plan_email(meal_plan: dict) -> str:
    """Format meal plan as readable email content (shopping list first)."""
    lines = []
    lines.append("=" * 40)
    lines.append(f"WEEKLY MEAL PLAN ({meal_plan['dietary_preference'].upper()})")
    lines.append("=" * 40)
    lines.append("")

    # Shopping list FIRST
    lines.append("SHOPPING LIST")
    lines.append("-" * 20)
    for item in meal_plan['shopping_list']:
        lines.append(f"- {item['total_quantity']} {item['name']}")
    lines.append("")

    # Recipes
    lines.append("=" * 40)
    lines.append("RECIPES")
    lines.append("=" * 40)
    for recipe in meal_plan['recipes']:
        lines.append(f"\n{recipe['day'].upper()} - {recipe['theme']}")
        lines.append(f"{recipe['dish_name']}")
        ingredients = ", ".join([f"{ing['quantity']} {ing['name']}" for ing in recipe['ingredients']])
        lines.append(f"Ingredients: {ingredients}")
        lines.append(f"Instructions: {recipe['instructions']}")

    return "\n".join(lines)


def send_meal_plan_email(recipient_email: str, meal_plan: dict) -> dict:
    """Send meal plan via email."""
    try:
        msg = MIMEMultipart()
        msg['From'] = EMAIL_SENDER
        msg['To'] = recipient_email
        msg['Subject'] = "Your Weekly Meal Plan"

        body = format_meal_plan_email(meal_plan)
        msg.attach(MIMEText(body, 'plain'))

        with smtplib.SMTP('smtp.gmail.com', 587) as server:
            server.starttls()
            server.login(EMAIL_SENDER, EMAIL_PASSWORD)
            server.send_message(msg)

        return {"success": True, "message": f"Email sent to {recipient_email}"}
    except Exception as e:
        return {"success": False, "message": f"Failed to send email: {str(e)}"}

## 7. Validation

In [8]:
def run_basic_validation(meal_plan_result: dict) -> dict:
    """Run quick programmatic validation checks (not LLM-based)."""
    if not meal_plan_result.get("success"):
        return {"passed": False, "errors": [meal_plan_result.get("error")]}

    meal_plan = meal_plan_result["meal_plan"]
    errors = []

    # Check 1: 7 recipes
    if len(meal_plan["recipes"]) != 7:
        errors.append(f"Expected 7 recipes, got {len(meal_plan['recipes'])}")

    # Check 2: Unique themes
    themes = [r["theme"].lower() for r in meal_plan["recipes"]]
    if len(themes) != len(set(themes)):
        duplicates = [t for t in themes if themes.count(t) > 1]
        errors.append(f"Duplicate themes: {set(duplicates)}")

    # Check 3: 5 ingredients per recipe
    for recipe in meal_plan["recipes"]:
        if len(recipe["ingredients"]) != 5:
            errors.append(f"{recipe['day']}: {len(recipe['ingredients'])} ingredients (expected 5)")

    return {"passed": len(errors) == 0, "errors": errors}

## 8. Gradio Interface

In [9]:
import gradio as gr
import time as pytime


def format_shopping_list(meal_plan: dict) -> str:
    """Format shopping list section."""
    lines = ["## Shopping List\n"]
    for item in meal_plan['shopping_list']:
        lines.append(f"- {item['total_quantity']} **{item['name']}**")
    return "\n".join(lines)


def format_single_recipe(recipe: dict) -> str:
    """Format a single recipe compactly."""
    ingredients = ", ".join([f"{ing['quantity']} {ing['name']}" for ing in recipe['ingredients']])
    return f"**{recipe['day']}** | {recipe['theme']} | *{recipe['dish_name']}*\n{ingredients}\n"


def generate_plan_streaming(dietary_pref, theme_requests, additional_prefs):
    """Generate meal plan with streaming output."""
    output = "Generating your meal plan..."
    yield output

    result = generator.generate(
        dietary_preference=dietary_pref,
        theme_requests=theme_requests if theme_requests.strip() else "random",
        additional_preferences=additional_prefs if additional_prefs.strip() else "none"
    )

    if not result["success"]:
        yield f"Error: {result['error']}"
        return

    meal_plan = result["meal_plan"]

    # Show shopping list first
    output = f"## Weekly {meal_plan['dietary_preference'].title()} Meal Plan\n\n"
    output += format_shopping_list(meal_plan) + "\n\n---\n\n"
    output += "## Recipes\n\n"
    yield output
    pytime.sleep(0.1)

    # Stream recipes one by one
    for recipe in meal_plan['recipes']:
        output += format_single_recipe(recipe) + "\n"
        yield output
        pytime.sleep(0.05)

    # Validation status
    validation = run_basic_validation(result)
    status = "All checks passed" if validation["passed"] else f"Issues: {', '.join(validation['errors'])}"
    output += f"---\n*Generated in {result['generation_time']}s | {status}*"
    yield output


def send_email_ui(email, dietary_pref, theme_requests, additional_prefs):
    """Generate and send meal plan via email."""
    if not email or "@" not in email:
        return "Enter a valid email address."

    result = generator.generate(
        dietary_preference=dietary_pref,
        theme_requests=theme_requests if theme_requests.strip() else "random",
        additional_preferences=additional_prefs if additional_prefs.strip() else "none"
    )

    if not result["success"]:
        return f"Generation failed: {result['error']}"

    email_result = send_meal_plan_email(email, result["meal_plan"])
    return email_result["message"]


# Create Gradio interface
with gr.Blocks(title="Weekly Meal Planner") as app:
    gr.Markdown("# Weekly Meal Planner\nGenerate 7 themed dinners with a consolidated shopping list.")

    with gr.Row():
        with gr.Column(scale=1):
            dietary_dropdown = gr.Dropdown(
                choices=["omnivore", "vegetarian", "vegan", "keto"],
                value="omnivore",
                label="Diet"
            )
            theme_input = gr.Textbox(label="Themes (optional)", placeholder="Italian, Mexican...", lines=1)
            additional_input = gr.Textbox(label="Preferences (optional)", placeholder="no nuts, quick meals...", lines=1)
            generate_btn = gr.Button("Generate", variant="primary")

            gr.Markdown("---")
            email_input = gr.Textbox(label="Email", placeholder="you@example.com")
            send_btn = gr.Button("Email Plan")
            email_status = gr.Textbox(label="Status", interactive=False)

        with gr.Column(scale=2):
            output_display = gr.Markdown("Click **Generate** to start.")

    generate_btn.click(
        fn=generate_plan_streaming,
        inputs=[dietary_dropdown, theme_input, additional_input],
        outputs=[output_display]
    )

    send_btn.click(
        fn=send_email_ui,
        inputs=[email_input, dietary_dropdown, theme_input, additional_input],
        outputs=[email_status]
    )

/opt/miniconda3/envs/bootcamp-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 9. Launch Application

In [10]:
# Launch the Gradio app
app.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://aa2667e4828e0639a8.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 10. Run Test Cases

Runs every test case through the generator and the basic validation checks. This makes 4 real API calls.

In [11]:
TEST_CASES = [
    {"dietary_preference": "omnivore", "theme_requests": "random", "additional_preferences": "none"},
    {"dietary_preference": "vegetarian", "theme_requests": "Italian, Asian, Mexican", "additional_preferences": "no mushrooms"},
    {"dietary_preference": "vegan", "theme_requests": "random", "additional_preferences": "high protein"},
    {"dietary_preference": "keto", "theme_requests": "Mediterranean, American", "additional_preferences": "none"}
]

for case in TEST_CASES:
    result = generator.generate(
        dietary_preference=case["dietary_preference"],
        theme_requests=case.get("theme_requests", "random"),
        additional_preferences=case.get("additional_preferences", "none")
    )
    validation = run_basic_validation(result)
    print(case["dietary_preference"], "->", "PASS" if validation["passed"] else validation["errors"])

omnivore -> PASS
vegetarian -> PASS
vegan -> PASS
keto -> PASS
